# Claude identification and screening

Normalize bibliographic exports and screen titles and abstracts.

Method and inputs: [workflow guide](../../docs/workflow.md).

In [1]:
import sys
from pathlib import Path

for _parent in (Path.cwd(), *Path.cwd().parents):
    if (_parent / "notebooks" / "_shared.py").is_file():
        sys.path.insert(0, str(_parent / "notebooks"))
        break
else:
    raise FileNotFoundError("Start Jupyter inside the repository.")

from _shared import (
    project_root, data_directory, results_directory, load_environment,
    require_environment, cache_directory, cache_fingerprint,
    load_cached_index, save_cached_index,
)

load_environment()

import os
from collections import deque
import csv
from anthropic import Anthropic
import pandas as pd
from pydantic import BaseModel, Field
import time
from typing import Annotated, Literal

folder_path = data_directory()
metadata_path = folder_path + 'metadata/'
result_path = results_directory()

ieee_file = metadata_path + 'ieee_validation.csv'
scopus_file = metadata_path + 'scopus_validation.csv'
wos_file = metadata_path + 'wos_validation.xls'

## IEEE Xplore

In [2]:
df_ieee = pd.read_csv(ieee_file)

df_ieee = df_ieee[pd.to_datetime(df_ieee['Date Added To Xplore'], errors='coerce') < pd.Timestamp('2024-07-17')]

df_ieee = df_ieee[['DOI', 'Document Title', 'Authors', 'Publication Year', 'Abstract', 'Article Citation Count']]
df_ieee = df_ieee.rename(columns={
    'Document Title': 'Title',
    'Publication Year': 'Year',
    'Article Citation Count': 'Citations'
})

df_ieee = df_ieee.dropna(subset=['DOI', 'Title', 'Abstract'])
df_ieee = df_ieee.loc[df_ieee['Year'] != 2025]
# df_ieee = df_ieee.dropna(subset=['Citations'])
df_ieee

,DOI,Title,Authors,Year,Abstract,Citations
0,10.1109/ACCESS.2024.3423462,MMW-AQA: Multimodal In-the-Wild Dataset for Ac...,T. Nagai; S. Takeda; S. Suzuki; H. Seshimo,2024,Action quality assessment (AQA) is a task for ...,1.0
1,10.23919/EUSIPCO55093.2022.9909680,Clotho-AQA: A Crowdsourced Dataset for Audio Q...,S. Lipping; P. Sudarsanam; K. Drossos; T. Virt...,2022,Audio question answering (AQA) is a multimodal...,20.0
2,10.1109/ICAACE61206.2024.10548995,Action Quality Assessment with Multi-scale Tem...,W. Wang; H. Wang; Y. Hao; Q. Wang,2024,Action quality assessment is a more challengin...,NaN
3,10.1109/ITME53901.2021.00048,Skeleton Based Action Quality Assessment of Fi...,H. -Y. Li; Q. Lei; H. -B. Zhang; J. -X. Du,2021,Action quality assessment(AQA) aims at achievi...,7.0
4,10.1109/ICIP42928.2021.9506257,Action Quality Assessment With Ignoring Scene ...,T. Nagai; S. Takeda; M. Matsumura; S. Shimizu;...,2021,We propose an action quality assessment (AQA) ...,6.0
5,10.1109/ICASSP48485.2024.10447069,Multi-Stage Contrastive Regression for Action ...,Q. An; M. Qi; H. Ma,2024,"In recent years, there has been growing intere...",7.0
6,10.1109/ICMLC58545.2023.10327994,Action Quality Assessment for ASD Behaviour Ev...,D. Zhang; D. Zhou; H. Liu,2023,Given the current increasing prevalence of aut...,2.0
8,10.1109/CVPR52729.2023.00238,LOGO: A Long-Form Video Dataset for Group Acti...,S. Zhang; W. Dai; S. Wang; X. Shen; J. Lu; J. ...,2023,Action quality assessment (AQA) has become an ...,34.0
9,10.1109/TIM.2024.3398072,Learning Sparse Temporal Video Mapping for Act...,S. Zahan; G. Mubashar Hassan; A. Mian,2024,Automated athlete performance measurement from...,5.0
10,10.1109/ICIP.2018.8451364,S3D: Stacking Segmental P3D for Action Quality...,X. Xiang; Y. Tian; A. Reiter; G. D. Hager; T. ...,2018,Action quality assessment is crucial in areas ...,45.0


### Total of 59 papers identified against 50 of the original study

## Scopus

In [3]:
df_scopus = pd.read_csv(scopus_file)

df_scopus = df_scopus[['DOI', 'Title', 'Authors', 'Year', 'Abstract', 'Cited by']]
df_scopus = df_scopus.rename(columns={
    'Cited by': 'Citations'
})

df_scopus = df_scopus.dropna(subset=['DOI', 'Title', 'Abstract'])
df_scopus = df_scopus.loc[df_scopus['Year'] != 2025]
# df_scopus = df_scopus.loc[df_scopus['Citations'] > 0]
df_scopus

,DOI,Title,Authors,Year,Abstract,Citations
0,10.1016/j.compbiomed.2024.108382,LightPRA: A Lightweight Temporal Convolutional...,"Sardari, S.; Sharifzadeh, S.; Daneshkhah, A.; ...",2024,Research evidence shows that physical rehabili...,7
1,10.1016/j.ins.2024.120347,Two-path target-aware contrastive regression f...,"Ke, X.; Xu, H.; Lin, X.; Guo, W.",2024,Action quality assessment (AQA) is a challengi...,11
2,10.1007/s10844-023-00824-x,Session-aware recommender system using double ...,"Khurana, P.; Gupta, B.; Sharma, R.; Bedi, P.",2024,Session-aware recommender systems capture user...,5
3,10.1007/s10489-024-05349-6,Assessing action quality with semantic-sequenc...,"Huang, F.; Li, J.",2024,Action Quality Assessment (AQA) is a critical ...,4
4,10.1109/WACV57701.2024.00641,Egocentric Action Recognition by Capturing Han...,"Shiota, T.; Takagi, M.; Kumagai, K.; Seshimo, ...",2024,Improving the performance of egocentric action...,6
...,...,...,...,...,...,...
134,10.1109/CVPRW.2017.16,Learning to Score Olympic Events,"Parmar, P.; Morris, B.",2017,"Estimating action quality, the process of assi...",164
135,10.1109/SSCI.2017.8285270,The open online repository of karate motion ca...,"Hachaj, T.; Ogiela, M.R.; Piekarczyk, M.",2017,The motivation of this paper is to introduce r...,12
136,10.1007/978-3-319-56357-2_19,Enhance performance of action evaluation funct...,"Huy, N.Q.; Nam, D.D.; Quoc, D.C.",2017,"In this paper, we describe how to optimize the...",0
137,10.1109/COMPSAC.2015.156,Context-aware framework for performance tuning...,"Nimalasena, A.; Getov, V.",2015,Context-aware systems perform adaptive changes...,2


In [4]:
dois_to_remove = [
    "10.14569/IJACSA.2024.0151119",
    "10.1109/PRAI62207.2024.10827404",
    "10.1109/MedAI62885.2024.00012",
    "10.1109/ICMRA62519.2024.10809303",
    "10.1109/IICAIET62352.2024.10730750",
    "10.1109/TIP.2024.3468870",
    "10.1007/978-981-97-7232-2_31",
    "10.1080/21680566.2024.2380913",
    "10.1109/TC.2024.3398503",
    "10.1007/s00521-023-09068-w",
    "10.1109/ICLT63507.2024.11038644",
    "10.1109/SWC62898.2024.00184"
]

df_scopus = df_scopus[~df_scopus["DOI"].isin(dois_to_remove)].reset_index(drop=True)
df_scopus

,DOI,Title,Authors,Year,Abstract,Citations
0,10.1016/j.compbiomed.2024.108382,LightPRA: A Lightweight Temporal Convolutional...,"Sardari, S.; Sharifzadeh, S.; Daneshkhah, A.; ...",2024,Research evidence shows that physical rehabili...,7
1,10.1016/j.ins.2024.120347,Two-path target-aware contrastive regression f...,"Ke, X.; Xu, H.; Lin, X.; Guo, W.",2024,Action quality assessment (AQA) is a challengi...,11
2,10.1007/s10844-023-00824-x,Session-aware recommender system using double ...,"Khurana, P.; Gupta, B.; Sharma, R.; Bedi, P.",2024,Session-aware recommender systems capture user...,5
3,10.1007/s10489-024-05349-6,Assessing action quality with semantic-sequenc...,"Huang, F.; Li, J.",2024,Action Quality Assessment (AQA) is a critical ...,4
4,10.1109/WACV57701.2024.00641,Egocentric Action Recognition by Capturing Han...,"Shiota, T.; Takagi, M.; Kumagai, K.; Seshimo, ...",2024,Improving the performance of egocentric action...,6
...,...,...,...,...,...,...
113,10.1109/CVPRW.2017.16,Learning to Score Olympic Events,"Parmar, P.; Morris, B.",2017,"Estimating action quality, the process of assi...",164
114,10.1109/SSCI.2017.8285270,The open online repository of karate motion ca...,"Hachaj, T.; Ogiela, M.R.; Piekarczyk, M.",2017,The motivation of this paper is to introduce r...,12
115,10.1007/978-3-319-56357-2_19,Enhance performance of action evaluation funct...,"Huy, N.Q.; Nam, D.D.; Quoc, D.C.",2017,"In this paper, we describe how to optimize the...",0
116,10.1109/COMPSAC.2015.156,Context-aware framework for performance tuning...,"Nimalasena, A.; Getov, V.",2015,Context-aware systems perform adaptive changes...,2


### Total of 118 papers identified against 119 of the original study

## Web of Science

In [5]:
df_wos = pd.read_excel(wos_file)

df_wos = df_wos[pd.to_datetime(df_wos['Indexed Date'], errors='coerce') < pd.Timestamp('2024-07-17')]

df_wos = df_wos[['DOI', 'Article Title', 'Authors', 'Publication Year', 'Abstract', 'Times Cited, All Databases']]
df_wos = df_wos.rename(columns={
    'Article Title': 'Title',
    'Publication Year': 'Year',
    'Times Cited, All Databases': 'Citations'
})

df_wos = df_wos.dropna(subset=['DOI', 'Title', 'Abstract'])
df_wos = df_wos.loc[df_wos['Year'] != 2025]
# df_wos = df_wos.loc[df_wos['Citations'] > 0]
df_wos

,DOI,Title,Authors,Year,Abstract,Citations
0,10.1109/TCSVT.2022.3143549,Semi-Supervised Action Quality Assessment With...,"Zhang, Shao-Jie; Pan, Jia-Hui; Gao, Jibin; Zhe...",2022,Action Quality Assessment aims to evaluate how...,26
2,10.1007/s00521-023-09068-w,Auto-encoding score distribution regression fo...,"Zhang, Boyu; Chen, Jiayuan; Xu, Yinfei; Zhang,...",2023,Assessing the quality of actions in videos is ...,20
3,10.1016/j.patrec.2022.04.015,Learning time-aware features for action qualit...,"Zhang, Yu; Xiong, Wei; Mi, Siya",2022,Action quality assessment (AQA) is a task to a...,8
5,10.1016/j.ins.2024.120347,Two-path target-aware contrastive regression f...,"Ke, Xiao; Xu, Huangbiao; Lin, Xiaofeng; Guo, W...",2024,Action quality assessment (AQA) is a challengi...,9
6,10.1007/s10489-022-03984-5,Label-reconstruction-based pseudo-subscore lea...,"Zhang, Hong-Bo; Dong, Li-Jia; Lei, Qing; Yang,...",2023,Most existing action quality assessment (AQA) ...,5
7,10.1007/s40747-022-00892-6,Gaussian guided frame sequence encoder network...,"Li, Ming-Zhe; Zhang, Hong-Bo; Dong, Li-Jia; Le...",2023,Can a computer evaluate an athlete's performan...,6
10,10.1109/TCSVT.2023.3281413,Hierarchical Graph Convolutional Networks for ...,"Zhou, Kanglei; Ma, Yue; Shum, Hubert P. H.; Li...",2023,Action quality assessment (AQA) automatically ...,41
11,10.1007/s10489-024-05349-6,Assessing action quality with semantic-sequenc...,"Huang, Feng; Li, Jianjun",2024,Action Quality Assessment (AQA) is a critical ...,5
12,10.1007/s10489-023-04613-5,Multi-skeleton structures graph convolutional ...,"Lei, Qing; Li, Huiying; Zhang, Hongbo; Du, Jix...",2023,In most existing action quality assessment (AQ...,8
13,10.3390/electronics11193051,An Efficient Motion Registration Method Based ...,"Ren, Yuhao; Zhang, Bochao; Chen, Jing; Guo, Li...",2022,Action quality assessment (AQA) is an importan...,0


### Total of 55 papers identified against 61 of the original paper

## Final Dataset

In [6]:
df_all = pd.concat([df_ieee, df_scopus, df_wos], ignore_index=True)
df_all

,DOI,Title,Authors,Year,Abstract,Citations
0,10.1109/ACCESS.2024.3423462,MMW-AQA: Multimodal In-the-Wild Dataset for Ac...,T. Nagai; S. Takeda; S. Suzuki; H. Seshimo,2024,Action quality assessment (AQA) is a task for ...,1.0
1,10.23919/EUSIPCO55093.2022.9909680,Clotho-AQA: A Crowdsourced Dataset for Audio Q...,S. Lipping; P. Sudarsanam; K. Drossos; T. Virt...,2022,Audio question answering (AQA) is a multimodal...,20.0
2,10.1109/ICAACE61206.2024.10548995,Action Quality Assessment with Multi-scale Tem...,W. Wang; H. Wang; Y. Hao; Q. Wang,2024,Action quality assessment is a more challengin...,NaN
3,10.1109/ITME53901.2021.00048,Skeleton Based Action Quality Assessment of Fi...,H. -Y. Li; Q. Lei; H. -B. Zhang; J. -X. Du,2021,Action quality assessment(AQA) aims at achievi...,7.0
4,10.1109/ICIP42928.2021.9506257,Action Quality Assessment With Ignoring Scene ...,T. Nagai; S. Takeda; M. Matsumura; S. Shimizu;...,2021,We propose an action quality assessment (AQA) ...,6.0
...,...,...,...,...,...,...
227,10.1007/s10844-023-00824-x,Session-aware recommender system using double ...,"Khurana, Purnima; Gupta, Bhavna; Sharma, Ravis...",2024,Session-aware recommender systems capture user...,6.0
228,10.1109/JIOT.2023.3301795,Efficient Massive-Device Orchestration Through...,"Shi, Haowei; Zou, Jiadao; Zhang, Qingxue",2024,Big data innovations are boosted by massive de...,2.0
229,10.1109/JSEN.2021.3051497,Dynamic Time Warping-Based Features With Class...,"Mohammadzade, Hoda; Hosseini, Soheil; Rezaei-D...",2021,This paper proposes a novel 3D action recognit...,22.0
230,10.1109/ACCESS.2022.3198935,Assessing Human Motion During Exercise Using M...,"Frangoudes, Fotos; Matsangidou, Maria; Schiza,...",2022,The World Health Organization promotes healthy...,14.0


### Total of 232 papers identified against 230 of the original study

In [7]:
df_all = df_all.drop_duplicates(subset=['DOI'])
df_all = df_all.drop(columns=['Citations'])
df_all = df_all.reset_index(drop=True)
df_all

,DOI,Title,Authors,Year,Abstract
0,10.1109/ACCESS.2024.3423462,MMW-AQA: Multimodal In-the-Wild Dataset for Ac...,T. Nagai; S. Takeda; S. Suzuki; H. Seshimo,2024,Action quality assessment (AQA) is a task for ...
1,10.23919/EUSIPCO55093.2022.9909680,Clotho-AQA: A Crowdsourced Dataset for Audio Q...,S. Lipping; P. Sudarsanam; K. Drossos; T. Virt...,2022,Audio question answering (AQA) is a multimodal...
2,10.1109/ICAACE61206.2024.10548995,Action Quality Assessment with Multi-scale Tem...,W. Wang; H. Wang; Y. Hao; Q. Wang,2024,Action quality assessment is a more challengin...
3,10.1109/ITME53901.2021.00048,Skeleton Based Action Quality Assessment of Fi...,H. -Y. Li; Q. Lei; H. -B. Zhang; J. -X. Du,2021,Action quality assessment(AQA) aims at achievi...
4,10.1109/ICIP42928.2021.9506257,Action Quality Assessment With Ignoring Scene ...,T. Nagai; S. Takeda; M. Matsumura; S. Shimizu;...,2021,We propose an action quality assessment (AQA) ...
...,...,...,...,...,...
140,10.1109/TCSVT.2020.3017727,Action Quality Assessment Using Siamese Networ...,"Jain, Hiteshi; Harit, Gaurav; Sharma, Avinash",2021,Automated vision-based score estimation models...
141,10.1007/s11042-012-1097-x,An audio-visual approach to web video categori...,"Ionescu, Bogdan Emanuel; Seyerlehner, Klaus; M...",2014,"In this paper, we discuss and audio-visual app..."
142,10.1155/2017/9281321,A Multicriteria Intelligence Aid Methodology U...,"Frini, Anissa",2017,Intelligence is increasingly relevant today in...
143,10.1038/s41558-018-0343-2,An assessment of climate action by high-carbon...,"Dietz, Simon; Fruitiere, Charles; Garcia-Manas...",2018,Corporations are an important source of GHG em...


### Total of 145 papers identified against 125 of the original study

## Step 2: Title + Abstract Evaluation with Claude

In [8]:
# Load environment variables

load_environment()

MODEL = os.getenv("MODEL_CLAUDE")
MODEL_SUPERVISOR = os.getenv("MODEL_SUPERVISOR_CLAUDE")
API_KEY = os.getenv("CLAUDE_API_KEY")

client = Anthropic(api_key=API_KEY)

In [9]:
# Rate limiter: max 2 requests every 60 seconds
_RATE_WINDOW_SEC = 60.0
_RATE_MAX_CALLS = 3
_rate_calls_log = deque()

batch_size = 15  # number of papers per batch

_sent_calls_count = 0
_completed_evaluations_count = 0

_total_input_tokens = 0
_total_output_tokens = 0

# Batch Evaluation

class BatchReviewItem(BaseModel):
    id: int
    decision: Annotated[Literal["yes", "no", "maybe"], Field(
        description="Decision about including the paper:\n"
                    "- 'yes': clear evidence that the article uses, shares, or discusses datasets, and it fully meets ALL inclusion criteria while violating NONE of the exclusion criteria.\n"
                    "- 'no': clear evidence that the article should be excluded because NOT ALL inclusion criteria are met or AT LEAST one exclusion criterion is violated.\n"
                    "- 'maybe': information is insufficient or ambiguous; full text reading is recommended.\n"
                    "If unsure, or it is needed to check other infos prefer 'maybe' over 'no'.")]
    reason: Annotated[str, Field(description="Short explanation for the decision.")]
    data_source: Annotated[str, Field(
        description="Official and uniquely identifiable name(s) of AQA dataset(s) explicitly used in the paper.\n"
                    "Guidelines:\n"
                    "- Populate this field ONLY if decision is 'yes'. Leave it empty otherwise.\n"
                    "- Include only real, unique, and identifiable resources (dataset, database, or data collection).\n"
                    "- If multiple datasets are mentioned, insert ALL of them in the same string separated by ';' (semicolon).\n"
                    "- Do NOT write generic descriptions (e.g., 'a large dataset', 'clinical dataset'); if the paper only describes the data generically, leave this field empty.")]


class BatchReview(BaseModel):
    items: list[BatchReviewItem]

fallback = BatchReviewItem(id=-1, decision="maybe", reason="No response", data_source="")


def is_open_data_relevant_batch(df_chunk: pd.DataFrame, client, model=None, max_retries=3):

    # Prepare record strings for prompt
    records = []
    for idx, row in df_chunk.iterrows():
        title = "" if pd.isna(row.get("Title")) else str(row.get("Title"))
        abstract = "" if pd.isna(row.get("Abstract")) else str(row.get("Abstract"))
        records.append(f"- id: {int(idx)}\n  Title: {title}\n  Abstract: {abstract}")

    # Build prompt with inclusion/exclusion criteria and instructions
    prompt = (
            "Task. Evaluate each record based on the following Inclusion and Exclusion Criteria and return ONLY JSON that matches the provided schema.\n\n"
            "Inclusion Criteria:\n"
            "(a) The article mentions the use or production of datasets, databases, or other data sources.\n"
            "(b) The article is related to computer science.\n"
            "Exclusion Criteria:\n"
            "(a) The article states that the data are not available, must be requested from the authors or are not free (cost is required).\n"
            "(b) The article presents results of surveys or review.\n"
            "(c) The article is not related to using vision-based methods.\n"
            "(d) The article is not related to Human Action Quality Assessment (AQA).\n"
            "Constraint. ALL the requirements should be met.\n\n"
            "Output Instructions:\n"
            "- For each provided record, return an object with: id, decision ('yes'|'no'|'maybe'), reason, data_source.\n"
            "- If title or abstract are missing or empty use decision='no' and reason='Title or abstract is missing', data_source=''.\n\n"
            "Records to evaluate:\n"
            + "\n".join(records)
    )

    global _sent_calls_count
    global _completed_evaluations_count
    global _total_input_tokens
    global _total_output_tokens

    backoff = 10

    # Error management with exponential backoff
    for _ in range(max_retries):
        _rate_limit_block_until_allowed()
        try:
            _sent_calls_count += 1
            resp = client.messages.parse(
                model=model,
                max_tokens=4096,
                temperature=0.0,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                output_format=BatchReview,
            )

            # Token Extraction
            _total_input_tokens += resp.usage.input_tokens
            _total_output_tokens += resp.usage.output_tokens

            # Update total evaluations count
            try:
                _completed_evaluations_count += len(df_chunk)
            except Exception as e:
                print(f"[attempt {_sent_calls_count}] CLAUDE error: {e}")
                _completed_evaluations_count = len(df_chunk)
            print(f"[CLAUDE] Completed {_completed_evaluations_count} total evaluations.")
            return resp.parsed_output.items
        except Exception as e:
            print(f"[attempt {_sent_calls_count}] CLAUDE error: {e}")
            time.sleep(backoff)
            backoff *= 2
    return []


def _rate_limit_block_until_allowed():
    # Block until the number of calls in the window is below max
    while True:
        now = time.monotonic()
        while _rate_calls_log and (now - _rate_calls_log[0]) >= _RATE_WINDOW_SEC:
            _rate_calls_log.popleft()
        if len(_rate_calls_log) < _RATE_MAX_CALLS:
            _rate_calls_log.append(now)
            return
        time.sleep(0.5)


# Process dataframe in batches

df_evaluated = df_all.copy()
start_total_time = time.monotonic()

for start in range(0, len(df_evaluated), batch_size):
    chunk = df_evaluated.iloc[start:start + batch_size]
    items = is_open_data_relevant_batch(chunk, client, model=MODEL)
    mapping = {int(it.id): it for it in items} if items else {}

    decisions = []
    reasons = []
    sources = []
    for idx in chunk.index:
        it = mapping.get(int(idx))
        if it is None:
            decisions.append(fallback.decision)
            reasons.append(fallback.reason)
            sources.append(fallback.data_source)
        else:
            decisions.append(it.decision)
            reasons.append(it.reason)
            sources.append(it.data_source)

    # Update dataframe with results
    df_evaluated.loc[chunk.index, 'Include'] = decisions
    df_evaluated.loc[chunk.index, 'Reason'] = reasons
    df_evaluated.loc[chunk.index, 'Data Source'] = sources

end_total_time = time.monotonic()
total_real_time_seconds = end_total_time - start_total_time
theoretical_min_time = ((_sent_calls_count - 1) // _RATE_MAX_CALLS) * _RATE_WINDOW_SEC if _sent_calls_count > 0 else 0

df_evaluated

[CLAUDE] Completed 15 total evaluations.
[CLAUDE] Completed 30 total evaluations.
[CLAUDE] Completed 45 total evaluations.
[CLAUDE] Completed 60 total evaluations.
[CLAUDE] Completed 75 total evaluations.
[CLAUDE] Completed 90 total evaluations.
[CLAUDE] Completed 105 total evaluations.
[CLAUDE] Completed 120 total evaluations.
[CLAUDE] Completed 135 total evaluations.
[CLAUDE] Completed 145 total evaluations.


,DOI,Title,Authors,Year,Abstract,Include,Reason,Data Source
0,10.1109/ACCESS.2024.3423462,MMW-AQA: Multimodal In-the-Wild Dataset for Ac...,T. Nagai; S. Takeda; S. Suzuki; H. Seshimo,2024,Action quality assessment (AQA) is a task for ...,yes,The article presents a new AQA dataset (MMW-AQ...,MMW-AQA
1,10.23919/EUSIPCO55093.2022.9909680,Clotho-AQA: A Crowdsourced Dataset for Audio Q...,S. Lipping; P. Sudarsanam; K. Drossos; T. Virt...,2022,Audio question answering (AQA) is a multimodal...,no,The article is about Audio Question Answering ...,
2,10.1109/ICAACE61206.2024.10548995,Action Quality Assessment with Multi-scale Tem...,W. Wang; H. Wang; Y. Hao; Q. Wang,2024,Action quality assessment is a more challengin...,maybe,The article discusses action quality assessmen...,
3,10.1109/ITME53901.2021.00048,Skeleton Based Action Quality Assessment of Fi...,H. -Y. Li; Q. Lei; H. -B. Zhang; J. -X. Du,2021,Action quality assessment(AQA) aims at achievi...,maybe,The article uses skeleton-based methods for ac...,
4,10.1109/ICIP42928.2021.9506257,Action Quality Assessment With Ignoring Scene ...,T. Nagai; S. Takeda; M. Matsumura; S. Shimizu;...,2021,We propose an action quality assessment (AQA) ...,maybe,The article proposes an AQA method evaluated o...,
...,...,...,...,...,...,...,...,...
140,10.1109/TCSVT.2020.3017727,Action Quality Assessment Using Siamese Networ...,"Jain, Hiteshi; Harit, Gaurav; Sharma, Avinash",2021,Automated vision-based score estimation models...,yes,Meets all inclusion criteria: uses Olympics Di...,
141,10.1007/s11042-012-1097-x,An audio-visual approach to web video categori...,"Ionescu, Bogdan Emanuel; Seyerlehner, Klaus; M...",2014,"In this paper, we discuss and audio-visual app...",no,Not related to Human Action Quality Assessment...,
142,10.1155/2017/9281321,A Multicriteria Intelligence Aid Methodology U...,"Frini, Anissa",2017,Intelligence is increasingly relevant today in...,no,Not related to Human Action Quality Assessment...,
143,10.1038/s41558-018-0343-2,An assessment of climate action by high-carbon...,"Dietz, Simon; Fruitiere, Charles; Garcia-Manas...",2018,Corporations are an important source of GHG em...,no,Not related to Human Action Quality Assessment...,


In [13]:
# --- FINAL REPORT AND COSTS ---

print("\n" + "="*40)
print("EXECUTION REPORT - PIPELINE")
print("="*40)

print(f"Papers Processed: {_completed_evaluations_count}")
print(f"API Calls made: {_sent_calls_count}")
print(f"Input Tokens consumed: {_total_input_tokens}")
print(f"Output Tokens generated: {_total_output_tokens}")
print(f"Execution Time: {total_real_time_seconds/60:.2f} min (Theoretical: {theoretical_min_time/60:.2f} min)")

efficiency_ratio = (theoretical_min_time / total_real_time_seconds * 100) if total_real_time_seconds > 0 else 0.0
print(f"Efficiency Ratio: {efficiency_ratio:.1f}%")

# Pricing CLAUDE Sonnet 4.5
PRICE_PER_1M_INPUT_TOKENS = 3
PRICE_PER_1M_OUTPUT_TOKENS = 15

cost_input = (_total_input_tokens / 1_000_000) * PRICE_PER_1M_INPUT_TOKENS
cost_output = (_total_output_tokens / 1_000_000) * PRICE_PER_1M_OUTPUT_TOKENS
cost_total = cost_input + cost_output

print("-" * 40)
print(f"Input Cost:  ${cost_input:.5f}")
print(f"Output Cost: ${cost_output:.5f}")
print(f"Estimated Total Cost: ${cost_total:.5f}")
print("="*40)


EXECUTION REPORT - PIPELINE
Papers Processed: 145
API Calls made: 10
Input Tokens consumed: 52903
Output Tokens generated: 9435
Execution Time: 3.34 min (Theoretical: 3.00 min)
Efficiency Ratio: 89.7%
----------------------------------------
Input Cost:  $0.15871
Output Cost: $0.14153
Estimated Total Cost: $0.30023


In [14]:
# Save run summary to CSV

SCREENING_DIR = result_path + "CLAUDE_screening"

def append_run_summary(filepath, row_dict):
    os.makedirs(os.path.dirname(os.path.abspath(filepath)), exist_ok=True)
    file_exists = os.path.isfile(filepath)
    with open(filepath, 'a', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=row_dict.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(row_dict)

append_run_summary(
    f"{SCREENING_DIR}/runs_cost_summary.csv",
    {
        "stage": "screening",
        "model": MODEL,
        "records_processed": _completed_evaluations_count,
        "api_calls": _sent_calls_count,
        "input_tokens": _total_input_tokens,
        "output_tokens": _total_output_tokens,
        "execution_time_sec": total_real_time_seconds,
        "theoretical_time_sec": theoretical_min_time,
        "cost_input_usd": cost_input,
        "cost_output_usd": cost_output,
        "cost_usd": cost_total
    }
)

In [12]:
# Review Process

_harm_calls_count = 0
_harm_input_tokens = 0
_harm_output_tokens = 0

class HarmonizedSourceRecord(BaseModel):
    id: int
    source: str


class HarmonizedSources(BaseModel):
    records: list[HarmonizedSourceRecord] = Field(
        description="Cleaned and harmonized data sources, one record for each input row."
    )


def harmonize_all_data_sources(df: pd.DataFrame, client, model=None, max_retries=3):
    global _harm_calls_count, _harm_input_tokens, _harm_output_tokens

    # Prepare inputs
    records = []
    expected_ids = list(range(len(df)))
    for row_id, (_, row) in enumerate(df.iterrows()):
        sources = "" if pd.isna(row.get("Data Source")) else str(row.get("Data Source"))
        records.append(f"- id: {row_id}\n  Data Source: {sources}")

    # Prompt
    prompt = (
        "Check and harmonize all dataset names for consistency and correctness.\n"
        "Guidelines:\n"
        "- Remove descriptors like 'dataset', 'database', 'data'. Also remove unnecessary adjectives like 'open', 'new'.\n"
        "- If multiple datasets are listed, leave the separation with ';' (semicolon) without spaces between different datasets.\n"
        "- Always capitalize the first letter of each dataset. If the name is all uppercase, leave it unchanged.\n"
        "- Prefer the recognized abbreviation if it exists, otherwise use the full name (e.g., 'Massachusetts Institute of Technology' becomes 'MIT', 'Piano Skills Assessment' becomes 'PISA').\n"
        "- Keep all actual dataset names; do NOT remove or discard any real dataset.\n"
        "- If a field is empty, keep it empty.\n\n"
        "Output Instructions:\n"
        "- Return only JSON matching the schema.\n"
        "- Return exactly one record per input row.\n"
        "- Keep the same id from each input row.\n\n"
        "Records to harmonize:\n"
        + "\n".join(records)
    )

    backoff = 10
    for _ in range(max_retries):
        _rate_limit_block_until_allowed()
        try:
            _harm_calls_count += 1
            resp = client.messages.parse(
                model=model,
                max_tokens=4096,
                temperature=0.0,
                messages=[
                    {"role": "user", "content": prompt}
                ],
                output_format=HarmonizedSources,
            )

            _harm_input_tokens += resp.usage.input_tokens
            _harm_output_tokens += resp.usage.output_tokens

            parsed_records = resp.parsed_output.records
            sources_by_id = {record.id: record.source for record in parsed_records}
            missing_ids = [row_id for row_id in expected_ids if row_id not in sources_by_id]
            extra_ids = [row_id for row_id in sources_by_id if row_id not in expected_ids]
            if len(parsed_records) != len(expected_ids) or missing_ids or extra_ids:
                raise ValueError(
                    f"Expected {len(expected_ids)} harmonized records, got {len(parsed_records)}. "
                    f"Missing ids: {missing_ids[:10]}; extra ids: {extra_ids[:10]}"
                )

            return pd.Series(
                [sources_by_id[row_id] for row_id in expected_ids],
                index=df.index,
                dtype="object",
            )
        except Exception as e:
            print(f"[CLAUDE ERROR] {e}")
            time.sleep(backoff)
            backoff *= 2
    return pd.Series("", index=df.index, dtype="object")

start_harm_time = time.monotonic()
df_evaluated["Data Source Harmonized"] = harmonize_all_data_sources(df_evaluated, client, model=MODEL_SUPERVISOR)
end_harm_time = time.monotonic()
total_harm_time_seconds = end_harm_time - start_harm_time

df_evaluated

,DOI,Title,Authors,Year,Abstract,Include,Reason,Data Source,Data Source Harmonized
0,10.1109/ACCESS.2024.3423462,MMW-AQA: Multimodal In-the-Wild Dataset for Ac...,T. Nagai; S. Takeda; S. Suzuki; H. Seshimo,2024,Action quality assessment (AQA) is a task for ...,yes,The article presents a new AQA dataset (MMW-AQ...,MMW-AQA,MMW-AQA
1,10.23919/EUSIPCO55093.2022.9909680,Clotho-AQA: A Crowdsourced Dataset for Audio Q...,S. Lipping; P. Sudarsanam; K. Drossos; T. Virt...,2022,Audio question answering (AQA) is a multimodal...,no,The article is about Audio Question Answering ...,,
2,10.1109/ICAACE61206.2024.10548995,Action Quality Assessment with Multi-scale Tem...,W. Wang; H. Wang; Y. Hao; Q. Wang,2024,Action quality assessment is a more challengin...,maybe,The article discusses action quality assessmen...,,
3,10.1109/ITME53901.2021.00048,Skeleton Based Action Quality Assessment of Fi...,H. -Y. Li; Q. Lei; H. -B. Zhang; J. -X. Du,2021,Action quality assessment(AQA) aims at achievi...,maybe,The article uses skeleton-based methods for ac...,,
4,10.1109/ICIP42928.2021.9506257,Action Quality Assessment With Ignoring Scene ...,T. Nagai; S. Takeda; M. Matsumura; S. Shimizu;...,2021,We propose an action quality assessment (AQA) ...,maybe,The article proposes an AQA method evaluated o...,,
...,...,...,...,...,...,...,...,...,...
140,10.1109/TCSVT.2020.3017727,Action Quality Assessment Using Siamese Networ...,"Jain, Hiteshi; Harit, Gaurav; Sharma, Avinash",2021,Automated vision-based score estimation models...,yes,Meets all inclusion criteria: uses Olympics Di...,,
141,10.1007/s11042-012-1097-x,An audio-visual approach to web video categori...,"Ionescu, Bogdan Emanuel; Seyerlehner, Klaus; M...",2014,"In this paper, we discuss and audio-visual app...",no,Not related to Human Action Quality Assessment...,,
142,10.1155/2017/9281321,A Multicriteria Intelligence Aid Methodology U...,"Frini, Anissa",2017,Intelligence is increasingly relevant today in...,no,Not related to Human Action Quality Assessment...,,
143,10.1038/s41558-018-0343-2,An assessment of climate action by high-carbon...,"Dietz, Simon; Fruitiere, Charles; Garcia-Manas...",2018,Corporations are an important source of GHG em...,no,Not related to Human Action Quality Assessment...,,


In [15]:
# --- FINAL REPORT AND COSTS (HARMONIZATION) ---

print("\n" + "="*40)
print("EXECUTION REPORT - HARMONIZATION")
print("="*40)

print(f"API Calls made: {_harm_calls_count}")
print(f"Input Tokens consumed: {_harm_input_tokens}")
print(f"Output Tokens generated: {_harm_output_tokens}")

# Pricing CLAUDE Opus 4.5
PRICE_PER_1M_INPUT_TOKENS = 5
PRICE_PER_1M_OUTPUT_TOKENS = 25

cost_input = (_harm_input_tokens / 1_000_000) * PRICE_PER_1M_INPUT_TOKENS
cost_output = (_harm_output_tokens / 1_000_000) * PRICE_PER_1M_OUTPUT_TOKENS
cost_total = cost_input + cost_output

print("-" * 40)
print(f"Input Cost:  ${cost_input:.5f}")
print(f"Output Cost: ${cost_output:.5f}")
print(f"Estimated Total Cost: ${cost_total:.5f}")
print("="*40)


EXECUTION REPORT - HARMONIZATION
API Calls made: 1
Input Tokens consumed: 2856
Output Tokens generated: 1707
----------------------------------------
Input Cost:  $0.01428
Output Cost: $0.04267
Estimated Total Cost: $0.05695


In [16]:
append_run_summary(
    f"{SCREENING_DIR}/runs_cost_summary.csv",
    {
        "stage": "harmonization",
        "model": MODEL_SUPERVISOR,
        "records_processed": len(df_evaluated),
        "api_calls": _harm_calls_count,
        "input_tokens": _harm_input_tokens,
        "output_tokens": _harm_output_tokens,
        "execution_time_sec": total_harm_time_seconds,
        "theoretical_time_sec": None,
        "cost_input_usd": cost_input,
        "cost_output_usd": cost_output,
        "cost_usd": cost_total
    }
)

In [17]:
df_evaluated_final = df_evaluated.copy()
df_evaluated_final["Data Source"] = df_evaluated_final["Data Source Harmonized"]
df_evaluated_final.drop(columns=["Data Source Harmonized"], inplace=True)

df_evaluated_final[['DOI', 'Include', 'Reason', 'Data Source']].to_csv(
    f"{SCREENING_DIR}/records.csv", index=True
)

In [18]:
# File Export
df_evaluated_final.to_csv(result_path + "CLAUDE_df_evaluated_validation.csv", index=False)